# Text Generation with RNN

This notebook implements a character/word-level Recurrent Neural Network (RNN) for text generation.
We train on **Frankenstein** by Mary Shelley from Project Gutenberg (public domain).

**Pipeline:**
1. Scrape raw text from Project Gutenberg
2. Preprocess & tokenize into sliding windows
3. Build & train an RNN (Embedding → RNN → Linear)
4. Generate new text from a seed sentence
5. Analyze the results

## 1. Imports & Device Setup

In [4]:
import re
import random
import requests
from bs4 import BeautifulSoup
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')

## 2. Dataset Preparation

### 2.1 Web Scraping

In [6]:
URL = 'https://www.gutenberg.org/cache/epub/84/pg84-images.html'

response = requests.get(URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

# Remove script / style tags
for tag in soup(['script', 'style', 'head']):
    tag.decompose()

raw_text = soup.get_text(separator=' ')

# Strip Gutenberg header / footer boilerplate
start_marker = 'FRANKENSTEIN'
end_marker = 'End of the Project Gutenberg'

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)

if start_idx != -1 and end_idx != -1:
    raw_text = raw_text[start_idx:end_idx]

print(f'Raw text length : {len(raw_text):,} characters')
print('\nFirst 500 characters:')
print(raw_text[:500])

Raw text length : 448,282 characters

First 500 characters:

 
 The Project Gutenberg eBook of  Frankenstein; or, the modern prometheus 
 This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at  www.gutenberg.org . If you are not located in the United States,
you will have to check the laws of the country where you are locat


### 2.2 Text Cleaning & Tokenization

In [7]:
def clean_text(text: str) -> str:
    """Lowercase and remove non-alphabetic characters (keep spaces)."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep only letters and whitespace
    text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces
    return text

clean = clean_text(raw_text)
tokens = clean.split() # word-level tokenization

print(f'Total tokens: {len(tokens):,}')
print(f'Unique tokens: {len(set(tokens)):,}')
print(f'\nFirst 20 tokens: {tokens[:20]}')

Total tokens: 78,399
Unique tokens: 7,259

First 20 tokens: ['the', 'project', 'gutenberg', 'ebook', 'of', 'frankenstein', 'or', 'the', 'modern', 'prometheus', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in']


### 2.3 Vocabulary

In [8]:
# Build vocabulary from most-frequent words (cap at MAX_VOCAB for efficiency)
MAX_VOCAB = 10_000
UNK_TOKEN = '<UNK>'

counter = Counter(tokens)
vocab = [UNK_TOKEN] + [w for w, _ in counter.most_common(MAX_VOCAB - 1)]

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

VOCAB_SIZE = len(vocab)
print(f'Vocabulary size: {VOCAB_SIZE:,}')

# Encode the full token stream
encoded = [word2idx.get(t, 0) for t in tokens]   # 0 as <UNK>
print(f'Encoded length: {len(encoded):,}')
print(f'First 10 indices: {encoded[:10]}')

Vocabulary size: 7,260
Encoded length: 78,399
First 10 indices: [1, 101, 87, 683, 4, 274, 35, 1, 601, 1618]


### 2.4 Sliding-Window Sequences

```
window_size = 100
input  shape: (N, 99)  99 context tokens
target shape: (N,)      1 next token per sample
```

In [10]:
window_size = 100 # specified in the assignment
seq_len = window_size - 1

data = [] # list of 99-token input sequences
targets = [] # list of single next-token targets

for i in range(len(encoded) - window_size + 1):
    data.append(encoded[i : i + seq_len])
    targets.append(encoded[i + seq_len])

print(f'Number of sequences: {len(data):,}')
print(f'Input sequence len: {len(data[0])}')
print(f'\nSample input: {data[0][:10]} ...')
print(f'Sample target: {targets[0]}')

Number of sequences: 78,300
Input sequence len: 99

Sample input: [1, 101, 87, 683, 4, 274, 35, 1, 601, 1618] ...
Sample target: 601


### 2.5 PyTorch Dataset & DataLoader

In [12]:
class TextDataset(Dataset):
    """
    Wraps pre-computed sliding-window sequences.

    Parameters
    ----------
    data : list of list of int
        Input token-index sequences, each of length seq_len.
    targets : list of int
        Next-token index for each sequence.
    """

    def __init__(self, data, targets):
        self.X = torch.tensor(data,    dtype=torch.long)
        self.y = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 128

dataset = TextDataset(data, targets)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Dataset size: {len(dataset):,}')
print(f'Batches/epoch: {len(dataloader):,}')

xb, yb = next(iter(dataloader))
print(f'Batch X shape: {xb.shape}')
print(f'Batch y shape: {yb.shape}')

Dataset size: 78,300
Batches/epoch: 612
Batch X shape: torch.Size([128, 99])
Batch y shape: torch.Size([128])


## 3. RNN Model

Architecture: **Embedding → RNN (with dropout) → Linear**

In [13]:
class TextRNN(nn.Module):
    """
    Word-level RNN for next-token prediction.

    Parameters
    ----------
    vocab_size  : int   – number of unique tokens (output classes)
    embed_dim   : int   – dimensionality of the token embedding
    hidden_dim  : int   – number of hidden units in each RNN layer
    num_layers  : int   – stacked RNN layers
    dropout     : float – dropout probability (applied between RNN layers)
    """

    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 num_layers=2, dropout=0.3):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # Recurrent layer
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            nonlinearity='tanh'
        )

        self.dropout = nn.Dropout(dropout)

        # Fully-connected output layer
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        """
        Parameters
        ----------
        x : Tensor (batch, seq_len)        – token indices
        h : Tensor (num_layers, batch, H)  – optional initial hidden state
        """
        emb = self.dropout(self.embedding(x))
        out, h_n = self.rnn(emb, h)
        last_out = out[:, -1, :]
        logits = self.fc(self.dropout(last_out))
        return logits, h_n

    def init_hidden(self, batch_size):
        """Return a zero hidden state on the correct device."""
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_dim, device=device)


# Hyper-parameters
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

model = TextRNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

TextRNN(
  (embedding): Embedding(7260, 128, padding_idx=0)
  (rnn): RNN(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=7260, bias=True)
)

Trainable parameters: 3,025,500


## 4. Training

In [ ]:
# Training config.
LEARNING_RATE = 3e-3
NUM_EPOCHS = 20
CLIP_GRAD = 5.0 # gradient clipping

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Training loop
history = {'loss': [], 'perplexity': []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits, _ = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()

        epoch_loss += loss.item() * x_batch.size(0)

    scheduler.step()

    avg_loss = epoch_loss / len(dataset)
    perplexity = np.exp(avg_loss)
    history['loss'].append(avg_loss)
    history['perplexity'].append(perplexity)

    print(f'Epoch {epoch:>2}/{NUM_EPOCHS}  '
          f'Loss: {avg_loss:.4f}  '
          f'Perplexity: {perplexity:.2f}  '
          f'LR: {scheduler.get_last_lr()[0]:.5f}')

print('\nTraining complete.')

### 4.1 Loss & Perplexity Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'], marker='o', color='steelblue')
axes[0].set_title('Training Loss (Cross-Entropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['perplexity'], marker='o', color='darkorange')
axes[1].set_title('Training Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Plot saved to training_curves.png')

In [ ]:
def generate_text(model, seed_sentence: str, num_words: int = 150,
                  temperature: float = 0.8) -> str:
    model.eval()

    seed_tokens = clean_text(seed_sentence).split()
    seed_idx = [word2idx.get(t, 0) for t in seed_tokens]

    # Pad or trim to exactly seq_len tokens
    if len(seed_idx) < seq_len:
        seed_idx = [0] * (seq_len - len(seed_idx)) + seed_idx
    else:
        seed_idx = seed_idx[-seq_len:] # keep last 99

    generated_words = list(seed_tokens) # for display
    context = seed_idx # sliding window buffer

    with torch.no_grad():
        for _ in range(num_words):
            x = torch.tensor([context], dtype=torch.long).to(device)
            logits, _ = model(x)

            # Temperature sampling
            probs = torch.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            generated_words.append(idx2word[next_idx])

            # Slide window forward by one
            context = context[1:] + [next_idx]

    return ' '.join(generated_words)


# Generate with different temperatures
SEED_SENTENCE = "I saw the pale student of unhallowed arts kneeling beside the thing he had put together"

for temp in [0.5, 0.8, 1.2]:
    print('\n')
    print(f'Temperature = {temp}')
    generated = generate_text(model, SEED_SENTENCE, num_words=100, temperature=temp)
    print(generated)

## 6. Analysis

### 6.1 Vocabulary Coverage of Generated Text

In [ ]:
sample = generate_text(model, SEED_SENTENCE, num_words=200, temperature=0.8)
sample_words = sample.split()

# Words that appear in the training corpus
corpus_set = set(tokens)
in_corpus = [w for w in sample_words if w in corpus_set]
coverage_pct = len(in_corpus) / len(sample_words) * 100

print(f'Generated words: {len(sample_words)}')
print(f'In-corpus words: {len(in_corpus)}')
print(f'Vocabulary coverage: {coverage_pct:.1f}%')

ttr = len(set(sample_words)) / len(sample_words)
print(f'Type-Token Ratio: {ttr:.3f}  (1.0 = all unique, 0.0 = all repeated)')

### 6.2 Most Frequent Generated Words

In [ ]:
gen_counter = Counter(sample_words)
top_generated = gen_counter.most_common(15)

words, counts = zip(*top_generated)

plt.figure(figsize=(10, 4))
plt.bar(words, counts, color='steelblue', edgecolor='white')
plt.title('Top 15 Most Frequent Words in Generated Text')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('generated_word_freq.png', dpi=150)
plt.show()